# Consolidação Multianual dos Dados

Este notebook consolida e valida os dados utilizados no projeto ao longo dos anos disponíveis.

Nesta etapa serão avaliados:

- arquivos do SIH/SUS entre 2021 e 2026;
- arquivos de leitos do CNES entre 2021 e 2026;
- fontes populacionais utilizadas para cada ano;
- quantidade de arquivos e meses disponíveis;
- consistência temporal das fontes;
- estrutura necessária para posterior integração dos datasets.

Os notebooks anteriores foram utilizados para compreender individualmente a estrutura de cada fonte. A partir desta etapa, o objetivo é trabalhar com a série histórica completa.

## 1. Importação das bibliotecas

In [257]:
from pathlib import Path
import pandas as pd

## 2. Configuração geral

Nesta etapa são definidos os anos analisados e os caminhos das fontes utilizadas no projeto.

In [258]:
ANOS = [
    2021,
    2022,
    2023,
    2024,
    2025,
    2026,
]

pasta_sih = Path("../data/raw/sih")
pasta_cnes = Path("../data/raw/cnes")
pasta_ibge = Path("../data/raw/ibge")

print("Anos analisados:", ANOS)
print("SIH:", pasta_sih.exists())
print("CNES:", pasta_cnes.exists())
print("IBGE:", pasta_ibge.exists())

Anos analisados: [2021, 2022, 2023, 2024, 2025, 2026]
SIH: True
CNES: True
IBGE: True


## 3. Inventário dos arquivos do SIH/SUS

Antes da consolidação dos dados, verificamos os arquivos mensais disponíveis para cada ano. Anos completos devem possuir normalmente 12 arquivos mensais. Para 2026, são considerados apenas os meses já disponíveis na fonte.

In [259]:
inventario_sih = []

for ano in ANOS:
    pasta_ano = pasta_sih / str(ano)

    arquivos = sorted(
        pasta_ano.glob("RDGO*.dbc")
    )

    for arquivo in arquivos:
        inventario_sih.append(
            {
                "ano": ano,
                "mes": arquivo.stem[-2:],
                "arquivo": arquivo.name,
                "tamanho_mb": round(
                    arquivo.stat().st_size / 1024**2,
                    2
                ),
            }
        )

inventario_sih = pd.DataFrame(inventario_sih)

inventario_sih.head()

,ano,mes,arquivo,tamanho_mb
0,2021,01,RDGO2101.dbc,1.89
1,2021,02,RDGO2102.dbc,1.82
2,2021,03,RDGO2103.dbc,2.01
3,2021,04,RDGO2104.dbc,1.88
4,2021,05,RDGO2105.dbc,2.00


In [260]:
resumo_sih = (
    inventario_sih
    .groupby("ano")
    .agg(
        arquivos=("arquivo", "count"),
        meses=("mes", lambda x: ", ".join(sorted(x))),
        tamanho_mb=("tamanho_mb", "sum"),
    )
    .reset_index()
)

resumo_sih

,ano,arquivos,meses,tamanho_mb
0,2021,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",23.75
1,2022,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",26.22
2,2023,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",29.76
3,2024,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",31.66
4,2025,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",33.24
5,2026,6,"01, 02, 03, 04, 05, 06",16.53


## 4. Verificação dos meses disponíveis no SIH/SUS

Nesta etapa verificamos se existem lacunas entre os meses esperados e os meses encontrados para cada ano.

In [261]:
for ano in ANOS:
    meses_encontrados = sorted(
        inventario_sih.loc[
            inventario_sih["ano"] == ano,
            "mes"
        ].tolist()
    )

    print(f"\n{ano}")
    print("Meses encontrados:", meses_encontrados)

    if not meses_encontrados:
        print("Nenhum arquivo encontrado.")
        continue

    ultimo_mes = int(max(meses_encontrados))

    meses_esperados = {
        f"{mes:02d}"
        for mes in range(1, ultimo_mes + 1)
    }

    faltantes = sorted(
        meses_esperados - set(meses_encontrados)
    )

    print(
        "Lacunas internas:",
        faltantes if faltantes else "Nenhuma"
    )

    if ano < 2026 and ultimo_mes < 12:
        print("Atenção: ano incompleto.")


2021
Meses encontrados: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Lacunas internas: Nenhuma

2022
Meses encontrados: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Lacunas internas: Nenhuma

2023
Meses encontrados: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Lacunas internas: Nenhuma

2024
Meses encontrados: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Lacunas internas: Nenhuma

2025
Meses encontrados: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Lacunas internas: Nenhuma

2026
Meses encontrados: ['01', '02', '03', '04', '05', '06']
Lacunas internas: Nenhuma


## 5. Inventário dos arquivos de leitos do CNES

Nesta etapa verificamos os arquivos mensais de leitos disponíveis para cada ano.

In [262]:
inventario_cnes = []

for ano in ANOS:
    pasta_ano = pasta_cnes / str(ano)

    arquivos = sorted( pasta_ano.glob("LTGO*.dbc"))

    for arquivo in arquivos:
        inventario_cnes.append(
            {
                "ano": ano,
                "mes": arquivo.stem[-2:],
                "arquivo": arquivo.name,
                "tamanho_mb": round(
                    arquivo.stat().st_size / 1024**2,
                    2
                ),
            }
        )

inventario_cnes = pd.DataFrame( inventario_cnes)
inventario_cnes.head()

,ano,mes,arquivo,tamanho_mb
0,2021,01,LTGO2101.dbc,0.03
1,2021,02,LTGO2102.dbc,0.03
2,2021,03,LTGO2103.dbc,0.03
3,2021,04,LTGO2104.dbc,0.03
4,2021,05,LTGO2105.dbc,0.03


In [263]:
resumo_cnes = (
    inventario_cnes
    .groupby("ano")
    .agg(
        arquivos=("arquivo", "count"),
        meses=("mes", lambda x: ", ".join(sorted(x))),
        tamanho_mb=("tamanho_mb", "sum"),
    )
    .reset_index()
)

resumo_cnes

,ano,arquivos,meses,tamanho_mb
0,2021,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",0.36
1,2022,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",0.36
2,2023,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",0.36
3,2024,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",0.36
4,2025,12,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12",0.36
5,2026,7,"01, 02, 03, 04, 05, 06, 07",0.21


## 6. Cobertura temporal do SIH/SUS e CNES

Para que capacidade hospitalar e registros de internação possam ser comparados é necessário verificar se as duas fontes possuem cobertura para as mesmas competências.

In [264]:
cobertura = (
    resumo_sih[["ano", "arquivos"]]
    .rename(
        columns={
            "arquivos": "arquivos_sih"
        }
    )
    .merge(
        resumo_cnes[
            ["ano", "arquivos"]
        ].rename(
            columns={
                "arquivos": "arquivos_cnes"
            }
        ),
        on="ano",
        how="outer"
    )
)

cobertura

,ano,arquivos_sih,arquivos_cnes
0,2021,12,12
1,2022,12,12
2,2023,12,12
3,2024,12,12
4,2025,12,12
5,2026,6,7


In [265]:
competencias_sih = set(
    zip(
        inventario_sih["ano"],
        inventario_sih["mes"]
    )
)

competencias_cnes = set(
    zip(
        inventario_cnes["ano"],
        inventario_cnes["mes"]
    )
)

somente_sih = sorted(
    competencias_sih - competencias_cnes
)

somente_cnes = sorted(
    competencias_cnes - competencias_sih
)

print("Competências somente no SIH:")
print(somente_sih)

print("\nCompetências somente no CNES:")
print(somente_cnes)

Competências somente no SIH:
[]

Competências somente no CNES:
[(2026, '07')]


## 7. Inventário das fontes populacionais

As informações populacionais utilizadas no projeto possuem diferentes produtos e metodologias conforme o ano. Por isso, as fontes são registradas separadamente antes da harmonização.

In [266]:
fontes_populacao = pd.DataFrame(
    [
        {
            "ano_referencia": 2021,
            "ano_publicacao": 2021,
            "tipo_dado": "Estimativa populacional",
            "origem": "IBGE",
            "arquivo": "estimativa_dou_2021.xls",
        },
        {
            "ano_referencia": 2022,
            "ano_publicacao": 2023,
            "tipo_dado": "População municipal publicada pelo IBGE para o TCU - base Censo 2022 / Malha 2023",
            "origem": "IBGE",
            "arquivo": "POP_TCU_2023_Municipios_POP2022_Malha2023.xls",
        },
        {
            "ano_referencia": 2023,
            "ano_publicacao": pd.NA,
            "tipo_dado": "Fonte populacional a definir",
            "origem": "A definir",
            "arquivo": "A definir",
        },
        {
            "ano_referencia": 2024,
            "ano_publicacao": 2024,
            "tipo_dado": "Estimativa populacional",
            "origem": "IBGE",
            "arquivo": "estimativa_dou_2024.xls",
        },
        {
            "ano_referencia": 2025,
            "ano_publicacao": 2025,
            "tipo_dado": "Estimativa populacional",
            "origem": "IBGE",
            "arquivo": "estimativa_dou_2025.xls",
        },
        {
            "ano_referencia": 2026,
            "ano_publicacao": pd.NA,
            "tipo_dado": "Fonte populacional a definir",
            "origem": "A definir",
            "arquivo": "A definir",
        },
    ]
)

fontes_populacao

,ano_referencia,ano_publicacao,tipo_dado,origem,arquivo
0,2021,2021,Estimativa populacional,IBGE,estimativa_dou_2021.xls
1,2022,2023,População municipal publicada pelo IBGE para o...,IBGE,POP_TCU_2023_Municipios_POP2022_Malha2023.xls
2,2023,<NA>,Fonte populacional a definir,A definir,A definir
3,2024,2024,Estimativa populacional,IBGE,estimativa_dou_2024.xls
4,2025,2025,Estimativa populacional,IBGE,estimativa_dou_2025.xls
5,2026,<NA>,Fonte populacional a definir,A definir,A definir


## 8. Arquivos populacionais disponíveis

Antes da harmonização da população municipal, verificamos os arquivos existentes em cada grupo de dados do IBGE.

In [267]:
arquivos_ibge = []

for arquivo in pasta_ibge.rglob("*"):
    if arquivo.is_file():
        arquivos_ibge.append(
            {
                "pasta": str(arquivo.parent.relative_to(pasta_ibge)),
                "arquivo": arquivo.name,
                "extensao": arquivo.suffix.lower(),
                "tamanho_mb": round(
                    arquivo.stat().st_size / 1024**2,
                    2
                ),
            }
        )

inventario_ibge = pd.DataFrame( arquivos_ibge).sort_values( ["pasta", "arquivo"])
inventario_ibge

,pasta,arquivo,extensao,tamanho_mb
0,censo_2022,Domicilios_52_publico.csv,.csv,39.98
1,censo_2022,Familia_52_publico.csv,.csv,18.01
2,censo_2022,Mortalidade_52_publico.csv,.csv,0.80
3,censo_2022,Pessoas_52_publico.csv,.csv,250.80
6,estimativas\2021,estimativa_dou_2021.xls,.xls,0.72
7,estimativas\2024,estimativa_dou_2024.xls,.xls,0.77
8,estimativas\2025,estimativa_dou_2025.xls,.xls,0.79
4,tcu_2023,POP_TCU_2023_Brasil_e_UFs_POP2022_Malha2023.xls,.xls,0.11
5,tcu_2023,POP_TCU_2023_Municipios_POP2022_Malha2023.xls,.xls,0.78


## 9. Validação das estimativas populacionais

Os arquivos de estimativas de 2021, 2024 e 2025 são verificados separadamente para identificar se possuem estrutura compatível.

In [268]:
arquivos_estimativas = {
    2021: pasta_ibge / "estimativas" / "2021" / "estimativa_dou_2021.xls",
    2024: pasta_ibge / "estimativas" / "2024" / "estimativa_dou_2024.xls",
    2025: pasta_ibge / "estimativas" / "2025" / "estimativa_dou_2025.xls",
}

for ano, arquivo in arquivos_estimativas.items():
    print(
        ano,
        arquivo.name,
        "→",
        arquivo.exists()
    )

2021 estimativa_dou_2021.xls → True
2024 estimativa_dou_2024.xls → True
2025 estimativa_dou_2025.xls → True


In [269]:
for ano, arquivo in arquivos_estimativas.items():
    excel = pd.ExcelFile(arquivo)

    print(f"\n{ano}")
    print("Planilhas:", excel.sheet_names)


2021
Planilhas: ['BRASIL E UFs', 'Municípios']

2024
Planilhas: ['BRASIL E UFs', 'MUNICÍPIOS']

2025
Planilhas: ['BRASIL E UFs', 'Municípios']


## 10. Estrutura das planilhas municipais das estimativas

Após confirmar a existência das planilhas municipais, verificamos sua estrutura antes de aplicar uma regra única de tratamento aos diferentes anos.

In [270]:
def localizar_aba_municipios(arquivo):
    excel = pd.ExcelFile(arquivo)

    abas = [
        aba
        for aba in excel.sheet_names
        if "munic" in aba.casefold()
    ]

    if len(abas) != 1:
        raise ValueError(
            f"Não foi possível identificar uma única aba municipal em {arquivo.name}: {abas}"
        )

    return abas[0]

In [271]:
for ano, arquivo in arquivos_estimativas.items():
    aba_municipios = localizar_aba_municipios(arquivo)

    print(f"\n{'=' * 60}")
    print(f"Ano: {ano}")
    print(f"Aba municipal: {aba_municipios}")
    print("=" * 60)

    amostra = pd.read_excel(
        arquivo,
        sheet_name=aba_municipios,
        header=None,
        nrows=15
    )

    display(amostra)


Ano: 2021
Aba municipal: Municípios


,0,1,2,3,4
0,ESTIMATIVAS DA POPULAÇÃO RESIDENTE NOS MUNICÍP...,NaN,NaN,NaN,NaN
1,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
2,RO,11,00015,Alta Floresta D'Oeste,22516
3,RO,11,00023,Ariquemes,111148
4,RO,11,00031,Cabixi,5067
5,RO,11,00049,Cacoal,86416
6,RO,11,00056,Cerejeiras,16088
7,RO,11,00064,Colorado do Oeste,15213
8,RO,11,00072,Corumbiara,7052
9,RO,11,00080,Costa Marques,19255



Ano: 2024
Aba municipal: MUNICÍPIOS


,0,1,2,3,4
0,ESTIMATIVAS DA POPULAÇÃO RESIDENTE NOS MUNICÍP...,NaN,NaN,NaN,NaN
1,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
2,RO,11,00015,Alta Floresta D'Oeste,22853
3,RO,11,00023,Ariquemes,108573
4,RO,11,00031,Cabixi,5690
5,RO,11,00049,Cacoal,97637
6,RO,11,00056,Cerejeiras,16975
7,RO,11,00064,Colorado do Oeste,16588
8,RO,11,00072,Corumbiara,8001
9,RO,11,00080,Costa Marques,13522



Ano: 2025
Aba municipal: Municípios


,0,1,2,3,4
0,ESTIMATIVAS DA POPULAÇÃO RESIDENTE NOS MUNICÍP...,NaN,NaN,NaN,NaN
1,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
2,RO,11,00015,Alta Floresta D'Oeste,22787
3,RO,11,00023,Ariquemes,109170
4,RO,11,00031,Cabixi,5664
5,RO,11,00049,Cacoal,98280
6,RO,11,00056,Cerejeiras,16966
7,RO,11,00064,Colorado do Oeste,16508
8,RO,11,00072,Corumbiara,7968
9,RO,11,00080,Costa Marques,13510


## 11. Padronização das estimativas populacionais

Como as estimativas de 2021, 2024 e 2025 apresentam a mesma estrutura municipal é aplicada uma única regra de leitura e padronização para os três anos.

In [272]:
def carregar_estimativa_ibge(arquivo, ano):
    aba_municipios = localizar_aba_municipios(arquivo)

    df = pd.read_excel(
        arquivo,
        sheet_name=aba_municipios,
        header=1,
        dtype={
            "COD. UF": "string",
            "COD. MUNIC": "string",
        }
    )

    df = df.rename(
        columns={
            "UF": "uf",
            "COD. UF": "cod_uf",
            "COD. MUNIC": "cod_municipio",
            "NOME DO MUNICÍPIO": "municipio",
            "POPULAÇÃO ESTIMADA": "populacao",
        }
    )

    # Padroniza os códigos utilizados para formar o código IBGE
    df["cod_uf"] = (
        df["cod_uf"]
        .str.strip()
        .str.zfill(2)
    )

    df["cod_municipio"] = (
        df["cod_municipio"]
        .str.strip()
        .str.zfill(5)
    )

    # Código IBGE completo com 7 dígitos
    df["codigo_ibge_7"] = (
        df["cod_uf"] + df["cod_municipio"]
    )

    # Trata a população preservando corretamente os valores numéricos
    populacao = (
        df["populacao"]
        .astype("string")
        .str.strip()
        .str.replace(r"\(\d+\)\s*$", "", regex=True)
        .str.strip()
    )

    # Remove ".0" gerado pela leitura de números do Excel
    populacao = populacao.str.replace(
        r"\.0$",
        "",
        regex=True
    )

    # Remove pontos utilizados como separadores de milhares
    populacao = populacao.str.replace(
        ".",
        "",
        regex=False
    )

    df["populacao"] = pd.to_numeric(
        populacao,
        errors="coerce"
    ).astype("Int64")

    # Mantém somente registros com código municipal válido
    df = df[
        df["codigo_ibge_7"].str.fullmatch(
            r"\d{7}",
            na=False
        )
    ].copy()

    # Metadados da fonte
    df["ano_referencia"] = ano
    df["ano_publicacao"] = ano
    df["origem"] = "IBGE"
    df["tipo_dado"] = "Estimativa populacional"

    return df

In [273]:
estimativas = []

for ano, arquivo in arquivos_estimativas.items():
    df_ano = carregar_estimativa_ibge(
        arquivo,
        ano
    )

    estimativas.append(df_ano)

df_estimativas = pd.concat(
    estimativas,
    ignore_index=True
)

df_estimativas.head()

,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7,ano_referencia,ano_publicacao,origem,tipo_dado
0,RO,11,00015,Alta Floresta D'Oeste,22516,1100015,2021,2021,IBGE,Estimativa populacional
1,RO,11,00023,Ariquemes,111148,1100023,2021,2021,IBGE,Estimativa populacional
2,RO,11,00031,Cabixi,5067,1100031,2021,2021,IBGE,Estimativa populacional
3,RO,11,00049,Cacoal,86416,1100049,2021,2021,IBGE,Estimativa populacional
4,RO,11,00056,Cerejeiras,16088,1100056,2021,2021,IBGE,Estimativa populacional


In [274]:
resumo_estimativas = (
    df_estimativas
    .groupby("ano_referencia")
    .agg(
        municipios=("codigo_ibge_7", "nunique"),
        populacao_total=("populacao", "sum"),
        nulos_populacao=("populacao", lambda x: x.isna().sum()),
    )
    .reset_index()
)

resumo_estimativas

,ano_referencia,municipios,populacao_total,nulos_populacao
0,2021,5570,213317639,0
1,2024,5570,212583750,0
2,2025,5571,213421037,0


In [275]:
resumo_go = (
    df_estimativas[
        df_estimativas["uf"] == "GO"
    ]
    .groupby("ano_referencia")
    .agg(
        municipios=("codigo_ibge_7", "nunique"),
        populacao_total=("populacao", "sum"),
    )
    .reset_index()
)

resumo_go

,ano_referencia,municipios,populacao_total
0,2021,246,7206589
1,2024,246,7350483
2,2025,246,7423629


### Validação da consolidação das estimativas

São verificadas duplicidades de código municipal, valores populacionais ausentes e códigos fora do padrão esperado em cada ano.

In [276]:
validacao_estimativas = (
    df_estimativas
    .groupby("ano_referencia")
    .apply(
        lambda df: pd.Series(
            {
                "registros": len(df),
                "codigos_unicos": df["codigo_ibge_7"].nunique(),
                "duplicados": df.duplicated(
                    subset=["codigo_ibge_7"]
                ).sum(),
                "populacao_nula": df["populacao"].isna().sum(),
                "codigos_invalidos": (
                    ~df["codigo_ibge_7"].str.fullmatch(
                        r"\d{7}",
                        na=False
                    )
                ).sum(),
            }
        ),
        include_groups=False
    )
    .reset_index()
)

validacao_estimativas

,ano_referencia,registros,codigos_unicos,duplicados,populacao_nula,codigos_invalidos
0,2021,5570,5570,0,0,0
1,2024,5570,5570,0,0,0
2,2025,5571,5571,0,0,0


## 12. Investigação da publicação TCU 2023 - população de referência 2022

O arquivo foi publicado em 2023 e utiliza a malha territorial de 2023, porém a informação populacional apresentada corresponde à população apurada pelo Censo Demográfico 2022.

Por isso, o ano de publicação e o ano de referência da população são tratados separadamente antes da harmonização.

In [277]:
arquivo_tcu_2023 = (
    pasta_ibge
    / "tcu_2023"
    / "POP_TCU_2023_Municipios_POP2022_Malha2023.xls"
)

print("Arquivo:", arquivo_tcu_2023.name)
print("Encontrado:", arquivo_tcu_2023.exists())

excel_tcu_2023 = pd.ExcelFile(arquivo_tcu_2023)

print("Planilhas:", excel_tcu_2023.sheet_names)

Arquivo: POP_TCU_2023_Municipios_POP2022_Malha2023.xls
Encontrado: True
Planilhas: ['Municípios']


In [278]:
for aba in excel_tcu_2023.sheet_names:
    print(f"\n{'=' * 60}")
    print("Planilha:", aba)
    print("=" * 60)

    amostra = pd.read_excel(
        arquivo_tcu_2023,
        sheet_name=aba,
        header=None,
        nrows=20
    )

    display(amostra)


Planilha: Municípios


,0,1,2,3,4
0,Relação da População dos Municípios para publi...,NaN,NaN,NaN,NaN
1,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2...
2,RO,11,00015,Alta Floresta D'Oeste,21494
3,RO,11,00023,Ariquemes,96833
4,RO,11,00031,Cabixi,5351
5,RO,11,00049,Cacoal,86887
6,RO,11,00056,Cerejeiras,15890
7,RO,11,00064,Colorado do Oeste,15663
8,RO,11,00072,Corumbiara,7519
9,RO,11,00080,Costa Marques,12627


In [279]:
df_tcu_bruto = pd.read_excel(
    arquivo_tcu_2023,
    sheet_name="Municípios",
    header=1,
    dtype={
        "COD. UF": "string",
        "COD. MUNIC": "string",
    }
)

print("Dimensão:", df_tcu_bruto.shape)

display(df_tcu_bruto.head())
display(df_tcu_bruto.tail(30))

Dimensão: (5603, 5)


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -
0,RO,11,00015,Alta Floresta D'Oeste,21494
1,RO,11,00023,Ariquemes,96833
2,RO,11,00031,Cabixi,5351
3,RO,11,00049,Cacoal,86887
4,RO,11,00056,Cerejeiras,15890


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -
5573,Notas:,<NA>,<NA>,NaN,NaN
5574,(1) População judicial do município de Porto V...,<NA>,<NA>,NaN,NaN
5575,(2) População judicial do município de Alvarãe...,<NA>,<NA>,NaN,NaN
5576,(3) População judicial do Município de Barcelo...,<NA>,<NA>,NaN,NaN
5577,(4) População judicial do município de Benjami...,<NA>,<NA>,NaN,NaN
5578,(5) População judicial do Município de Boca do...,<NA>,<NA>,NaN,NaN
5579,(6) População judicial do Município de Borba-A...,<NA>,<NA>,NaN,NaN
5580,(7) População judicial do município de Caapira...,<NA>,<NA>,NaN,NaN
5581,(8) População judicial do Município de Codajás...,<NA>,<NA>,NaN,NaN
5582,(9) População judicial do município de Eirunep...,<NA>,<NA>,NaN,NaN


In [280]:
print("Colunas:")

for coluna in df_tcu_bruto.columns:
    print(repr(coluna))

Colunas:
'UF'
'COD. UF'
'COD. MUNIC'
'NOME DO MUNICÍPIO'
'POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -'


### Identificação dos registros municipais válidos

O arquivo contém notas e observações após os registros municipais. Por isso, os códigos são analisados antes da remoção dessas linhas, permitindo identificar quantos registros correspondem efetivamente a municípios.

In [281]:
df_tcu_teste = df_tcu_bruto.copy()

df_tcu_teste["cod_uf"] = (
    df_tcu_teste["COD. UF"]
    .astype("string")
    .str.strip()
    .str.zfill(2)
)

df_tcu_teste["cod_municipio"] = (
    df_tcu_teste["COD. MUNIC"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

df_tcu_teste["codigo_ibge_7"] = (
    df_tcu_teste["cod_uf"]
    + df_tcu_teste["cod_municipio"]
)

mascara_codigo_valido = (
    df_tcu_teste["codigo_ibge_7"]
    .str.fullmatch(r"\d{7}", na=False)
)

print("Linhas totais:", len(df_tcu_teste))
print("Registros com código válido:", mascara_codigo_valido.sum())
print("Registros fora do padrão:", (~mascara_codigo_valido).sum())

Linhas totais: 5603
Registros com código válido: 5570
Registros fora do padrão: 33


In [282]:
df_tcu_validos_teste = df_tcu_teste[ mascara_codigo_valido].copy()

print( "Códigos únicos:", df_tcu_validos_teste["codigo_ibge_7"].nunique())

print(
    "Códigos duplicados:",
    df_tcu_validos_teste.duplicated(
        subset=["codigo_ibge_7"]
    ).sum()
)

Códigos únicos: 5570
Códigos duplicados: 0


In [283]:
display(
    df_tcu_validos_teste[
        [
            "UF",
            "cod_uf",
            "cod_municipio",
            "codigo_ibge_7",
            "NOME DO MUNICÍPIO",
        ]
    ].tail(15)
)

,UF,cod_uf,cod_municipio,codigo_ibge_7,NOME DO MUNICÍPIO
5555,GO,52,21403,5221403,Trindade
5556,GO,52,21452,5221452,Trombas
5557,GO,52,21502,5221502,Turvânia
5558,GO,52,21551,5221551,Turvelândia
5559,GO,52,21577,5221577,Uirapuru
5560,GO,52,21601,5221601,Uruaçu
5561,GO,52,21700,5221700,Uruana
5562,GO,52,21809,5221809,Urutaí
5563,GO,52,21858,5221858,Valparaíso de Goiás
5564,GO,52,21908,5221908,Varjão


In [284]:
df_tcu_go_teste = df_tcu_validos_teste[
    df_tcu_validos_teste["UF"] == "GO"
].copy()

print(
    "Municípios de Goiás:",
    df_tcu_go_teste["codigo_ibge_7"].nunique()
)

display(
    df_tcu_go_teste.head()
)

Municípios de Goiás: 246


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -,cod_uf,cod_municipio,codigo_ibge_7
5323,GO,52,00050,Abadia de Goiás,19128,52,00050,5200050
5324,GO,52,00100,Abadiânia,17232,52,00100,5200100
5325,GO,52,00134,Acreúna,21568,52,00134,5200134
5326,GO,52,00159,Adelândia,2297,52,00159,5200159
5327,GO,52,00175,Água Fria de Goiás,4954,52,00175,5200175


In [285]:
coluna_pop_tcu = (
    "POPULAÇÃO APURADA IBGE \n"
    "- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -"
)

go_com_observacao = df_tcu_go_teste[
    df_tcu_go_teste[coluna_pop_tcu]
    .astype("string")
    .str.contains(r"\(\d+\)", regex=True, na=False)
]

print(
    "Municípios de Goiás com observação na população:",
    len(go_com_observacao)
)

display(go_com_observacao)

Municípios de Goiás com observação na população: 0


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -,cod_uf,cod_municipio,codigo_ibge_7


### Verificação dos registros descartados

Os registros sem código municipal válido são inspecionados para confirmar que correspondem apenas a linhas auxiliares, totais, observações ou notas da publicação.

In [286]:
df_tcu_invalidos = df_tcu_teste[
    ~mascara_codigo_valido
].copy()

print(
    "Total de registros descartados:",
    len(df_tcu_invalidos)
)

display(
    df_tcu_invalidos.head(10)
)

Total de registros descartados: 33


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -,cod_uf,cod_municipio,codigo_ibge_7
5570,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5571,Fonte: IBGE. Diretoria de Pesquisas - DPE - C...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5572,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5573,Notas:,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5574,(1) População judicial do município de Porto V...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5575,(2) População judicial do município de Alvarãe...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5576,(3) População judicial do Município de Barcelo...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5577,(4) População judicial do município de Benjami...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5578,(5) População judicial do Município de Boca do...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5579,(6) População judicial do Município de Borba-A...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>


In [287]:
display(
    df_tcu_invalidos.tail(35)
)

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -,cod_uf,cod_municipio,codigo_ibge_7
5570,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5571,Fonte: IBGE. Diretoria de Pesquisas - DPE - C...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5572,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5573,Notas:,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5574,(1) População judicial do município de Porto V...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5575,(2) População judicial do município de Alvarãe...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5576,(3) População judicial do Município de Barcelo...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5577,(4) População judicial do município de Benjami...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5578,(5) População judicial do Município de Boca do...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
5579,(6) População judicial do Município de Borba-A...,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>


## 13. Padronização da população de referência 2022

Após a validação estrutural da publicação do IBGE para o TCU, os registros municipais são tratados e padronizados para a mesma estrutura utilizada pelas demais fontes populacionais.

In [288]:
def carregar_populacao_tcu_2022(arquivo):
    df = pd.read_excel(
        arquivo,
        sheet_name="Municípios",
        header=1,
        dtype={
            "COD. UF": "string",
            "COD. MUNIC": "string",
        }
    )

    coluna_populacao = (
        "POPULAÇÃO APURADA IBGE \n"
        "- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -"
    )

    df = df.rename(
        columns={
            "UF": "uf",
            "COD. UF": "cod_uf",
            "COD. MUNIC": "cod_municipio",
            "NOME DO MUNICÍPIO": "municipio",
            coluna_populacao: "populacao",
        }
    )

    df["cod_uf"] = (
        df["cod_uf"]
        .astype("string")
        .str.strip()
        .str.zfill(2)
    )

    df["cod_municipio"] = (
        df["cod_municipio"]
        .astype("string")
        .str.strip()
        .str.zfill(5)
    )

    df["codigo_ibge_7"] = (
        df["cod_uf"] + df["cod_municipio"]
    )

    # Mantém somente registros municipais
    df = df[
        df["codigo_ibge_7"].str.fullmatch(
            r"\d{7}",
            na=False
        )
    ].copy()

    # Remove marcadores de notas e separadores de milhares
    populacao = (
        df["populacao"]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\(\d+\)\s*$",
            "",
            regex=True
        )
        .str.strip()
        .str.replace(
            ".",
            "",
            regex=False
        )
    )

    df["populacao"] = pd.to_numeric(
        populacao,
        errors="coerce"
    ).astype("Int64")

    df["ano_referencia"] = 2022
    df["ano_publicacao"] = 2023
    df["origem"] = "IBGE"
    df["tipo_dado"] = (
        "População municipal publicada pelo IBGE para o TCU "
        "- base Censo 2022 / Malha 2023"
    )

    return df

In [289]:
df_pop_2022 = carregar_populacao_tcu_2022(
    arquivo_tcu_2023
)

df_pop_2022.head()

,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7,ano_referencia,ano_publicacao,origem,tipo_dado
0,RO,11,00015,Alta Floresta D'Oeste,21494,1100015,2022,2023,IBGE,População municipal publicada pelo IBGE para o...
1,RO,11,00023,Ariquemes,96833,1100023,2022,2023,IBGE,População municipal publicada pelo IBGE para o...
2,RO,11,00031,Cabixi,5351,1100031,2022,2023,IBGE,População municipal publicada pelo IBGE para o...
3,RO,11,00049,Cacoal,86887,1100049,2022,2023,IBGE,População municipal publicada pelo IBGE para o...
4,RO,11,00056,Cerejeiras,15890,1100056,2022,2023,IBGE,População municipal publicada pelo IBGE para o...


In [290]:
validacao_2022 = pd.DataFrame(
    {
        "registros": [len(df_pop_2022)],
        "codigos_unicos": [
            df_pop_2022["codigo_ibge_7"].nunique()
        ],
        "duplicados": [
            df_pop_2022.duplicated(
                subset=["codigo_ibge_7"]
            ).sum()
        ],
        "populacao_nula": [
            df_pop_2022["populacao"].isna().sum()
        ],
        "codigos_invalidos": [
            (
                ~df_pop_2022["codigo_ibge_7"]
                .str.fullmatch(r"\d{7}", na=False)
            ).sum()
        ],
    }
)

validacao_2022

,registros,codigos_unicos,duplicados,populacao_nula,codigos_invalidos
0,5570,5570,0,0,0


In [291]:
resumo_go_2022 = (
    df_pop_2022[
        df_pop_2022["uf"] == "GO"
    ]
    .agg(
        municipios=("codigo_ibge_7", "nunique"),
        populacao_total=("populacao", "sum"),
    )
)

resumo_go_2022

,codigo_ibge_7,populacao
municipios,246.0,NaN
populacao_total,NaN,7056495.0


## 14. Consolidação das fontes populacionais disponíveis


In [292]:
df_populacao = pd.concat(
    [
        df_estimativas,
        df_pop_2022,
    ],
    ignore_index=True
)

df_populacao.head()

,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7,ano_referencia,ano_publicacao,origem,tipo_dado
0,RO,11,00015,Alta Floresta D'Oeste,22516,1100015,2021,2021,IBGE,Estimativa populacional
1,RO,11,00023,Ariquemes,111148,1100023,2021,2021,IBGE,Estimativa populacional
2,RO,11,00031,Cabixi,5067,1100031,2021,2021,IBGE,Estimativa populacional
3,RO,11,00049,Cacoal,86416,1100049,2021,2021,IBGE,Estimativa populacional
4,RO,11,00056,Cerejeiras,16088,1100056,2021,2021,IBGE,Estimativa populacional


In [293]:
resumo_populacao_go = (
    df_populacao[
        df_populacao["uf"] == "GO"
    ]
    .groupby("ano_referencia")
    .agg(
        municipios=("codigo_ibge_7", "nunique"),
        populacao_total=("populacao", "sum"),
    )
    .reset_index()
)

resumo_populacao_go

,ano_referencia,municipios,populacao_total
0,2021,246,7206589
1,2022,246,7056495
2,2024,246,7350483
3,2025,246,7423629


## 15. Cobertura temporal da população

Nesta etapa são identificados os anos que já possuem uma fonte populacional validada e os anos que ainda exigem definição metodológica antes da construção dos indicadores per capita.

In [294]:
anos_populacao = set(
    df_populacao["ano_referencia"]
    .dropna()
    .astype(int)
)

anos_sem_populacao = sorted( set(ANOS) - anos_populacao)
print( "Anos com população validada:", sorted(anos_populacao))
print( "Anos ainda sem população definida:", anos_sem_populacao)

Anos com população validada: [2021, 2022, 2024, 2025]
Anos ainda sem população definida: [2023, 2026]


## 16. Validação do schema multianual do SIH/SUS

Antes da consolidação dos registros hospitalares, verificamos se os arquivos mensais do SIH/SUS mantêm a mesma estrutura de colunas entre 2021 e 2026.

In [295]:
from pysus.api.extensions import ExtensionFactory

In [296]:
arquivos_sih_multianual = []

for ano in ANOS:
    pasta_ano = pasta_sih / str(ano)

    arquivos_sih_multianual.extend(
        sorted( pasta_ano.glob("RDGO*.dbc"))
    )

print( "Total de arquivos SIH:", len(arquivos_sih_multianual))
print( "Primeiro arquivo:", arquivos_sih_multianual[0].name)
print( "Último arquivo:", arquivos_sih_multianual[-1].name)

Total de arquivos SIH: 66
Primeiro arquivo: RDGO2101.dbc
Último arquivo: RDGO2606.dbc


In [297]:
schema_sih = []

colunas_referencia = None

for i, arquivo in enumerate(
    arquivos_sih_multianual,
    start=1
):
    print(
        f"[{i:02d}/{len(arquivos_sih_multianual)}] "
        f"{arquivo.name}"
    )

    ext = await ExtensionFactory.instantiate(
        arquivo
    )

    df_temp = await ext.load()

    colunas = list(df_temp.columns)

    if colunas_referencia is None:
        colunas_referencia = colunas.copy()

    conjunto_referencia = set(
        colunas_referencia
    )

    conjunto_atual = set(colunas)

    colunas_faltantes = sorted(
        conjunto_referencia
        - conjunto_atual
    )

    colunas_extras = sorted(
        conjunto_atual
        - conjunto_referencia
    )

    schema_sih.append(
        {
            "ano": int(arquivo.parent.name),
            "mes": int(arquivo.stem[-2:]),
            "arquivo": arquivo.name,
            "registros": len(df_temp),
            "n_colunas": len(colunas),
            "mesmas_colunas": (
                conjunto_atual
                == conjunto_referencia
            ),
            "mesma_ordem": (
                colunas
                == colunas_referencia
            ),
            "colunas_faltantes": colunas_faltantes,
            "colunas_extras": colunas_extras,
        }
    )

    del df_temp
    del ext

[01/66] RDGO2101.dbc
[02/66] RDGO2102.dbc
[03/66] RDGO2103.dbc
[04/66] RDGO2104.dbc
[05/66] RDGO2105.dbc
[06/66] RDGO2106.dbc
[07/66] RDGO2107.dbc
[08/66] RDGO2108.dbc
[09/66] RDGO2109.dbc
[10/66] RDGO2110.dbc
[11/66] RDGO2111.dbc
[12/66] RDGO2112.dbc
[13/66] RDGO2201.dbc
[14/66] RDGO2202.dbc
[15/66] RDGO2203.dbc
[16/66] RDGO2204.dbc
[17/66] RDGO2205.dbc
[18/66] RDGO2206.dbc
[19/66] RDGO2207.dbc
[20/66] RDGO2208.dbc
[21/66] RDGO2209.dbc
[22/66] RDGO2210.dbc
[23/66] RDGO2211.dbc
[24/66] RDGO2212.dbc
[25/66] RDGO2301.dbc
[26/66] RDGO2302.dbc
[27/66] RDGO2303.dbc
[28/66] RDGO2304.dbc
[29/66] RDGO2305.dbc
[30/66] RDGO2306.dbc
[31/66] RDGO2307.dbc
[32/66] RDGO2308.dbc
[33/66] RDGO2309.dbc
[34/66] RDGO2310.dbc
[35/66] RDGO2311.dbc
[36/66] RDGO2312.dbc
[37/66] RDGO2401.dbc
[38/66] RDGO2402.dbc
[39/66] RDGO2403.dbc
[40/66] RDGO2404.dbc
[41/66] RDGO2405.dbc
[42/66] RDGO2406.dbc
[43/66] RDGO2407.dbc
[44/66] RDGO2408.dbc
[45/66] RDGO2409.dbc
[46/66] RDGO2410.dbc
[47/66] RDGO2411.dbc
[48/66] RDGO2

In [298]:
df_schema_sih = pd.DataFrame( schema_sih)
df_schema_sih.head()

,ano,mes,arquivo,registros,n_colunas,mesmas_colunas,mesma_ordem,colunas_faltantes,colunas_extras
0,2021,1,RDGO2101.dbc,27074,113,True,True,[],[]
1,2021,2,RDGO2102.dbc,26092,113,True,True,[],[]
2,2021,3,RDGO2103.dbc,29333,113,True,True,[],[]
3,2021,4,RDGO2104.dbc,28169,113,True,True,[],[]
4,2021,5,RDGO2105.dbc,29498,113,True,True,[],[]


In [299]:
resumo_schema_sih = (
    df_schema_sih
    .groupby("ano")
    .agg(
        arquivos=("arquivo", "count"),
        registros=("registros", "sum"),
        min_colunas=("n_colunas", "min"),
        max_colunas=("n_colunas", "max"),
        schemas_diferentes=(
            "mesmas_colunas",
            lambda x: (~x).sum()
        ),
        ordem_diferente=(
            "mesma_ordem",
            lambda x: (~x).sum()
        ),
    )
    .reset_index()
)

resumo_schema_sih

,ano,arquivos,registros,min_colunas,max_colunas,schemas_diferentes,ordem_diferente
0,2021,12,343686,113,113,0,0
1,2022,12,369947,113,113,0,0
2,2023,12,423932,113,113,0,0
3,2024,12,452002,113,113,0,0
4,2025,12,467902,113,114,10,10
5,2026,6,229286,114,114,6,6


In [300]:
problemas_schema_sih = df_schema_sih[
    (~df_schema_sih["mesmas_colunas"])
    | (~df_schema_sih["mesma_ordem"])
].copy()

print(
    "Arquivos com diferença de schema:",
    len(problemas_schema_sih)
)

display(
    problemas_schema_sih
)

Arquivos com diferença de schema: 16


,ano,mes,arquivo,registros,n_colunas,mesmas_colunas,mesma_ordem,colunas_faltantes,colunas_extras
50,2025,3,RDGO2503.dbc,40265,114,False,False,[],[FONTE_ORC]
51,2025,4,RDGO2504.dbc,39034,114,False,False,[],[FONTE_ORC]
52,2025,5,RDGO2505.dbc,41650,114,False,False,[],[FONTE_ORC]
53,2025,6,RDGO2506.dbc,39560,114,False,False,[],[FONTE_ORC]
54,2025,7,RDGO2507.dbc,39900,114,False,False,[],[FONTE_ORC]
55,2025,8,RDGO2508.dbc,39314,114,False,False,[],[FONTE_ORC]
56,2025,9,RDGO2509.dbc,38427,114,False,False,[],[FONTE_ORC]
57,2025,10,RDGO2510.dbc,38979,114,False,False,[],[FONTE_ORC]
58,2025,11,RDGO2511.dbc,38223,114,False,False,[],[FONTE_ORC]
59,2025,12,RDGO2512.dbc,37196,114,False,False,[],[FONTE_ORC]


In [301]:
print(
    "Número de colunas de referência:",
    len(colunas_referencia)
)

for coluna in colunas_referencia:
    print(coluna)

Número de colunas de referência: 113
UF_ZI
ANO_CMPT
MES_CMPT
ESPEC
CGC_HOSP
N_AIH
IDENT
CEP
MUNIC_RES
NASC
SEXO
UTI_MES_IN
UTI_MES_AN
UTI_MES_AL
UTI_MES_TO
MARCA_UTI
UTI_INT_IN
UTI_INT_AN
UTI_INT_AL
UTI_INT_TO
DIAR_ACOM
QT_DIARIAS
PROC_SOLIC
PROC_REA
VAL_SH
VAL_SP
VAL_SADT
VAL_RN
VAL_ACOMP
VAL_ORTP
VAL_SANGUE
VAL_SADTSR
VAL_TRANSP
VAL_OBSANG
VAL_PED1AC
VAL_TOT
VAL_UTI
US_TOT
DT_INTER
DT_SAIDA
DIAG_PRINC
DIAG_SECUN
COBRANCA
NATUREZA
NAT_JUR
GESTAO
RUBRICA
IND_VDRL
MUNIC_MOV
COD_IDADE
IDADE
DIAS_PERM
MORTE
NACIONAL
NUM_PROC
CAR_INT
TOT_PT_SP
CPF_AUT
HOMONIMO
NUM_FILHOS
INSTRU
CID_NOTIF
CONTRACEP1
CONTRACEP2
GESTRISCO
INSC_PN
SEQ_AIH5
CBOR
CNAER
VINCPREV
GESTOR_COD
GESTOR_TP
GESTOR_CPF
GESTOR_DT
CNES
CNPJ_MANT
INFEHOSP
CID_ASSO
CID_MORTE
COMPLEX
FINANC
FAEC_TP
REGCT
RACA_COR
ETNIA
SEQUENCIA
REMESSA
AUD_JUST
SIS_JUST
VAL_SH_FED
VAL_SP_FED
VAL_SH_GES
VAL_SP_GES
VAL_UCI
MARCA_UCI
DIAGSEC1
DIAGSEC2
DIAGSEC3
DIAGSEC4
DIAGSEC5
DIAGSEC6
DIAGSEC7
DIAGSEC8
DIAGSEC9
TPDISEC1
TPDISEC2
TPDISEC3
TPDIS

In [302]:
resumo_schema_sih

,ano,arquivos,registros,min_colunas,max_colunas,schemas_diferentes,ordem_diferente
0,2021,12,343686,113,113,0,0
1,2022,12,369947,113,113,0,0
2,2023,12,423932,113,113,0,0
3,2024,12,452002,113,113,0,0
4,2025,12,467902,113,114,10,10
5,2026,6,229286,114,114,6,6


### Resultado da validação

Os arquivos do SIH/SUS mantêm 113 colunas entre janeiro de 2021 e fevereiro de 2025. A partir de março de 2025 foi adicionada a coluna `FONTE_ORC`, totalizando 114 colunas.

Nenhuma coluna da estrutura de referência foi removida. Como `FONTE_ORC` não é necessária para os indicadores previstos nesta etapa, a consolidação multianual utilizará apenas o conjunto de campos selecionados para a análise.

## 17. Consolidação multianual do SIH/SUS

Após a validação do schema, os arquivos mensais são processados utilizando apenas os campos necessários para as etapas posteriores de tratamento e análise.

In [303]:
colunas_sih_selecionadas = [
    "ANO_CMPT",
    "MES_CMPT",
    "N_AIH",
    "IDENT",
    "SEQ_AIH5",
    "CNES",
    "MUNIC_RES",
    "MUNIC_MOV",
    "DT_INTER",
    "DT_SAIDA",
    "DIAS_PERM",
    "UTI_MES_TO",
    "VAL_TOT",
    "DIAG_PRINC",
    "MORTE",
    "ESPEC",
    "PROC_REA",
    "CAR_INT",
]

In [304]:
colunas_ausentes_sih = sorted(
    set(colunas_sih_selecionadas)
    - set(colunas_referencia)
)

print(
    "Campos selecionados ausentes no schema de referência:",
    len(colunas_ausentes_sih)
)

print(
    "Colunas ausentes:",
    colunas_ausentes_sih
)

Campos selecionados ausentes no schema de referência: 0
Colunas ausentes: []


### Consolidação dos registros 

In [305]:
dados_sih = []

for i, arquivo in enumerate(
    arquivos_sih_multianual,
    start=1
):
    print(
        f"[{i:02d}/{len(arquivos_sih_multianual)}] "
        f"{arquivo.name}"
    )

    ext = await ExtensionFactory.instantiate(
        arquivo
    )

    df_temp = await ext.load()

    df_temp = df_temp[
        colunas_sih_selecionadas
    ].copy()

    df_temp["ARQUIVO_ORIGEM"] = arquivo.name

    dados_sih.append(df_temp)

    del ext
    del df_temp

[01/66] RDGO2101.dbc
[02/66] RDGO2102.dbc
[03/66] RDGO2103.dbc
[04/66] RDGO2104.dbc
[05/66] RDGO2105.dbc
[06/66] RDGO2106.dbc
[07/66] RDGO2107.dbc
[08/66] RDGO2108.dbc
[09/66] RDGO2109.dbc
[10/66] RDGO2110.dbc
[11/66] RDGO2111.dbc
[12/66] RDGO2112.dbc
[13/66] RDGO2201.dbc
[14/66] RDGO2202.dbc
[15/66] RDGO2203.dbc
[16/66] RDGO2204.dbc
[17/66] RDGO2205.dbc
[18/66] RDGO2206.dbc
[19/66] RDGO2207.dbc
[20/66] RDGO2208.dbc
[21/66] RDGO2209.dbc
[22/66] RDGO2210.dbc
[23/66] RDGO2211.dbc
[24/66] RDGO2212.dbc
[25/66] RDGO2301.dbc
[26/66] RDGO2302.dbc
[27/66] RDGO2303.dbc
[28/66] RDGO2304.dbc
[29/66] RDGO2305.dbc
[30/66] RDGO2306.dbc
[31/66] RDGO2307.dbc
[32/66] RDGO2308.dbc
[33/66] RDGO2309.dbc
[34/66] RDGO2310.dbc
[35/66] RDGO2311.dbc
[36/66] RDGO2312.dbc
[37/66] RDGO2401.dbc
[38/66] RDGO2402.dbc
[39/66] RDGO2403.dbc
[40/66] RDGO2404.dbc
[41/66] RDGO2405.dbc
[42/66] RDGO2406.dbc
[43/66] RDGO2407.dbc
[44/66] RDGO2408.dbc
[45/66] RDGO2409.dbc
[46/66] RDGO2410.dbc
[47/66] RDGO2411.dbc
[48/66] RDGO2

In [306]:
df_sih_multianual = pd.concat( dados_sih, ignore_index=True)
print( "Dimensão consolidada:", df_sih_multianual.shape)
df_sih_multianual.head()

Dimensão consolidada: (2286755, 19)


,ANO_CMPT,MES_CMPT,N_AIH,IDENT,SEQ_AIH5,CNES,MUNIC_RES,MUNIC_MOV,DT_INTER,DT_SAIDA,DIAS_PERM,UTI_MES_TO,VAL_TOT,DIAG_PRINC,MORTE,ESPEC,PROC_REA,CAR_INT,ARQUIVO_ORIGEM
0,2021,01,5220103703310,1,000,6665322,521930,521930,2020-10-29,20201101,3,0,485.78,K929,0,03,0303070102,02,RDGO2101.dbc
1,2021,01,5220103703320,1,000,6665322,521930,521930,2020-10-31,20201103,3,0,242.88,R31,0,03,0303150050,02,RDGO2101.dbc
2,2021,01,5220103703353,1,000,6665322,521930,521930,2020-10-08,20201009,1,0,40.38,I743,0,03,0301060070,02,RDGO2101.dbc
3,2021,01,5220103703397,1,000,6665322,521300,521930,2020-10-28,20201102,5,0,503.85,C819,0,03,0304100021,02,RDGO2101.dbc
4,2021,01,5220103703419,1,000,6665322,521930,521930,2020-11-11,20201112,1,0,40.38,K922,1,03,0301060070,02,RDGO2101.dbc


In [307]:
#validação
resumo_sih_multianual = (
    df_sih_multianual
    .groupby("ANO_CMPT")
    .size()
    .reset_index(
        name="registros"
    )
)

resumo_sih_multianual

,ANO_CMPT,registros
0,2021,343686
1,2022,369947
2,2023,423932
3,2024,452002
4,2025,467902
5,2026,229286


## 18. Validação do schema multianual do CNES

Antes da consolidação dos dados de capacidade hospitalar, verificamos se os arquivos mensais de leitos do CNES mantêm a mesma estrutura de colunas entre 2021 e 2026.

In [308]:
arquivos_cnes_multianual = []

for ano in ANOS:
    pasta_ano = pasta_cnes / str(ano)

    arquivos_cnes_multianual.extend(
        sorted(
            pasta_ano.glob("LTGO*.dbc")
        )
    )

print( "Total de arquivos CNES:", len(arquivos_cnes_multianual))
print( "Primeiro arquivo:", arquivos_cnes_multianual[0].name)
print( "Último arquivo:", arquivos_cnes_multianual[-1].name)

Total de arquivos CNES: 67
Primeiro arquivo: LTGO2101.dbc
Último arquivo: LTGO2607.dbc


In [309]:
schema_cnes = []

colunas_referencia_cnes = None

for i, arquivo in enumerate( arquivos_cnes_multianual, start=1):
    print(
        f"[{i:02d}/{len(arquivos_cnes_multianual)}] "
        f"{arquivo.name}"
    )

    # Carrega o arquivo mensal
    ext = await ExtensionFactory.instantiate(
        arquivo
    )

    df_temp = await ext.load()

    colunas = list(df_temp.columns)

    if colunas_referencia_cnes is None:
        colunas_referencia_cnes = colunas.copy()

    conjunto_referencia = set(
        colunas_referencia_cnes
    )

    conjunto_atual = set(colunas)

    colunas_faltantes = sorted(
        conjunto_referencia
        - conjunto_atual
    )

    colunas_extras = sorted(
        conjunto_atual
        - conjunto_referencia
    )

    schema_cnes.append(
        {
            "ano": int(arquivo.parent.name),
            "mes": int(arquivo.stem[-2:]),
            "arquivo": arquivo.name,
            "registros": len(df_temp),
            "n_colunas": len(colunas),
            "mesmas_colunas": (
                conjunto_atual
                == conjunto_referencia
            ),
            "mesma_ordem": (
                colunas
                == colunas_referencia_cnes
            ),
            "colunas_faltantes": colunas_faltantes,
            "colunas_extras": colunas_extras,
        }
    )

    del ext
    del df_temp

[01/67] LTGO2101.dbc
[02/67] LTGO2102.dbc
[03/67] LTGO2103.dbc
[04/67] LTGO2104.dbc
[05/67] LTGO2105.dbc
[06/67] LTGO2106.dbc
[07/67] LTGO2107.dbc
[08/67] LTGO2108.dbc
[09/67] LTGO2109.dbc
[10/67] LTGO2110.dbc
[11/67] LTGO2111.dbc
[12/67] LTGO2112.dbc
[13/67] LTGO2201.dbc
[14/67] LTGO2202.dbc
[15/67] LTGO2203.dbc
[16/67] LTGO2204.dbc
[17/67] LTGO2205.dbc
[18/67] LTGO2206.dbc
[19/67] LTGO2207.dbc
[20/67] LTGO2208.dbc
[21/67] LTGO2209.dbc
[22/67] LTGO2210.dbc
[23/67] LTGO2211.dbc
[24/67] LTGO2212.dbc
[25/67] LTGO2301.dbc
[26/67] LTGO2302.dbc
[27/67] LTGO2303.dbc
[28/67] LTGO2304.dbc
[29/67] LTGO2305.dbc
[30/67] LTGO2306.dbc
[31/67] LTGO2307.dbc
[32/67] LTGO2308.dbc
[33/67] LTGO2309.dbc
[34/67] LTGO2310.dbc
[35/67] LTGO2311.dbc
[36/67] LTGO2312.dbc
[37/67] LTGO2401.dbc
[38/67] LTGO2402.dbc
[39/67] LTGO2403.dbc
[40/67] LTGO2404.dbc
[41/67] LTGO2405.dbc
[42/67] LTGO2406.dbc
[43/67] LTGO2407.dbc
[44/67] LTGO2408.dbc
[45/67] LTGO2409.dbc
[46/67] LTGO2410.dbc
[47/67] LTGO2411.dbc
[48/67] LTGO2

In [310]:
df_schema_cnes = pd.DataFrame(
    schema_cnes
)

df_schema_cnes.head()

,ano,mes,arquivo,registros,n_colunas,mesmas_colunas,mesma_ordem,colunas_faltantes,colunas_extras
0,2021,1,LTGO2101.dbc,2895,28,True,True,[],[]
1,2021,2,LTGO2102.dbc,2889,28,True,True,[],[]
2,2021,3,LTGO2103.dbc,2922,28,True,True,[],[]
3,2021,4,LTGO2104.dbc,2956,28,True,True,[],[]
4,2021,5,LTGO2105.dbc,2965,28,True,True,[],[]


In [311]:
resumo_schema_cnes = (
    df_schema_cnes
    .groupby("ano")
    .agg(
        arquivos=("arquivo", "count"),
        registros=("registros", "sum"),
        min_colunas=("n_colunas", "min"),
        max_colunas=("n_colunas", "max"),
        schemas_diferentes=(
            "mesmas_colunas",
            lambda x: (~x).sum()
        ),
        ordem_diferente=(
            "mesma_ordem",
            lambda x: (~x).sum()
        ),
    )
    .reset_index()
)

resumo_schema_cnes

,ano,arquivos,registros,min_colunas,max_colunas,schemas_diferentes,ordem_diferente
0,2021,12,35793,28,28,0,0
1,2022,12,36003,28,28,0,0
2,2023,12,35288,28,28,0,0
3,2024,12,35190,28,28,0,0
4,2025,12,35522,28,28,0,0
5,2026,7,20652,28,28,0,0


In [312]:
problemas_schema_cnes = df_schema_cnes[
    (~df_schema_cnes["mesmas_colunas"])
    | (~df_schema_cnes["mesma_ordem"])
].copy()

print(
    "Arquivos com diferença de schema:",
    len(problemas_schema_cnes)
)

display(
    problemas_schema_cnes
)

Arquivos com diferença de schema: 0


,ano,mes,arquivo,registros,n_colunas,mesmas_colunas,mesma_ordem,colunas_faltantes,colunas_extras


In [313]:
print( "Número de colunas de referência:", len(colunas_referencia_cnes))
for coluna in colunas_referencia_cnes:
    print(coluna)

Número de colunas de referência: 28
CNES
CODUFMUN
REGSAUDE
MICR_REG
DISTRSAN
DISTRADM
TPGESTAO
PF_PJ
CPF_CNPJ
NIV_DEP
CNPJ_MAN
ESFERA_A
ATIVIDAD
RETENCAO
NATUREZA
CLIENTEL
TP_UNID
TURNO_AT
NIV_HIER
TERCEIRO
TP_LEITO
CODLEITO
QT_EXIST
QT_CONTR
QT_SUS
QT_NSUS
COMPETEN
NAT_JUR


## 19. Consolidação multianual do CNES

Após a validação do schema, os arquivos mensais de leitos são consolidados utilizando os campos necessários para representar a capacidade hospitalar disponível em cada competência.

In [314]:
colunas_cnes_selecionadas = [
    "CNES",
    "CODUFMUN",
    "TP_UNID",
    "TP_LEITO",
    "CODLEITO",
    "QT_EXIST",
    "QT_SUS",
    "QT_NSUS",
    "COMPETEN",
]

In [315]:
colunas_ausentes_cnes = sorted( set(colunas_cnes_selecionadas) - set(colunas_referencia_cnes))
print( "Campos selecionados ausentes no schema de referência:",len(colunas_ausentes_cnes))
print( "Colunas ausentes:", colunas_ausentes_cnes)

Campos selecionados ausentes no schema de referência: 0
Colunas ausentes: []


In [316]:
dados_cnes = []

for i, arquivo in enumerate(
    arquivos_cnes_multianual,
    start=1
):
    print(
        f"[{i:02d}/{len(arquivos_cnes_multianual)}] "
        f"{arquivo.name}"
    )

    # Carrega o arquivo mensal
    ext = await ExtensionFactory.instantiate(
        arquivo
    )

    df_temp = await ext.load()

    # Mantém as colunas selecionadas
    df_temp = df_temp[
        colunas_cnes_selecionadas
    ].copy()

    # Preserva a origem e a competência
    df_temp["ARQUIVO_ORIGEM"] = arquivo.name
    df_temp["ANO"] = int(arquivo.parent.name)
    df_temp["MES"] = int(arquivo.stem[-2:])

    dados_cnes.append(df_temp)

    del ext
    del df_temp

[01/67] LTGO2101.dbc
[02/67] LTGO2102.dbc
[03/67] LTGO2103.dbc
[04/67] LTGO2104.dbc


[05/67] LTGO2105.dbc
[06/67] LTGO2106.dbc
[07/67] LTGO2107.dbc
[08/67] LTGO2108.dbc
[09/67] LTGO2109.dbc
[10/67] LTGO2110.dbc
[11/67] LTGO2111.dbc
[12/67] LTGO2112.dbc
[13/67] LTGO2201.dbc
[14/67] LTGO2202.dbc
[15/67] LTGO2203.dbc
[16/67] LTGO2204.dbc
[17/67] LTGO2205.dbc
[18/67] LTGO2206.dbc
[19/67] LTGO2207.dbc
[20/67] LTGO2208.dbc
[21/67] LTGO2209.dbc
[22/67] LTGO2210.dbc
[23/67] LTGO2211.dbc
[24/67] LTGO2212.dbc
[25/67] LTGO2301.dbc
[26/67] LTGO2302.dbc
[27/67] LTGO2303.dbc
[28/67] LTGO2304.dbc
[29/67] LTGO2305.dbc
[30/67] LTGO2306.dbc
[31/67] LTGO2307.dbc
[32/67] LTGO2308.dbc
[33/67] LTGO2309.dbc
[34/67] LTGO2310.dbc
[35/67] LTGO2311.dbc
[36/67] LTGO2312.dbc
[37/67] LTGO2401.dbc
[38/67] LTGO2402.dbc
[39/67] LTGO2403.dbc
[40/67] LTGO2404.dbc
[41/67] LTGO2405.dbc
[42/67] LTGO2406.dbc
[43/67] LTGO2407.dbc
[44/67] LTGO2408.dbc
[45/67] LTGO2409.dbc
[46/67] LTGO2410.dbc
[47/67] LTGO2411.dbc
[48/67] LTGO2412.dbc
[49/67] LTGO2501.dbc
[50/67] LTGO2502.dbc
[51/67] LTGO2503.dbc
[52/67] LTGO2

In [317]:
df_cnes_multianual = pd.concat( dados_cnes, ignore_index=True)
print( "Dimensão consolidada:", df_cnes_multianual.shape )
df_cnes_multianual.head()

Dimensão consolidada: (198448, 12)


,CNES,CODUFMUN,TP_UNID,TP_LEITO,CODLEITO,QT_EXIST,QT_SUS,QT_NSUS,COMPETEN,ARQUIVO_ORIGEM,ANO,MES
0,9331603,520010,15,2,33,9,9,0,202101,LTGO2101.dbc,2021,1
1,2335506,520013,05,6,34,4,3,1,202101,LTGO2101.dbc,2021,1
2,2335506,520013,05,1,03,2,1,1,202101,LTGO2101.dbc,2021,1
3,2335506,520013,05,5,45,3,3,0,202101,LTGO2101.dbc,2021,1
4,2335506,520013,05,4,43,3,3,0,202101,LTGO2101.dbc,2021,1


In [318]:
colunas_quantidade_cnes = [
    "QT_EXIST",
    "QT_SUS",
    "QT_NSUS",
]

for coluna in colunas_quantidade_cnes:
    df_cnes_multianual[coluna] = pd.to_numeric(
        df_cnes_multianual[coluna],
        errors="coerce"
    )

In [319]:
resumo_cnes_multianual = (
    df_cnes_multianual
    .groupby("ANO")
    .agg(
        registros=("CNES", "size"),
        meses=("MES", "nunique"),
    )
    .reset_index()
)

resumo_cnes_multianual

,ANO,registros,meses
0,2021,35793,12
1,2022,36003,12
2,2023,35288,12
3,2024,35190,12
4,2025,35522,12
5,2026,20652,7


In [320]:
duplicados_chave_cnes = (
    df_cnes_multianual
    .duplicated(
        subset=[
            "CNES",
            "TP_LEITO",
            "CODLEITO",
            "COMPETEN",
        ]
    )
    .sum()
)

print(
    "Duplicados na chave CNES + tipo + leito + competência:",
    duplicados_chave_cnes
)

Duplicados na chave CNES + tipo + leito + competência: 0


In [321]:
inconsistencias_leitos = (
    df_cnes_multianual["QT_EXIST"]
    != (
        df_cnes_multianual["QT_SUS"]
        + df_cnes_multianual["QT_NSUS"]
    )
).sum()

print(
    "Registros com inconsistência entre leitos existentes, SUS e não SUS:",
    inconsistencias_leitos
)

Registros com inconsistência entre leitos existentes, SUS e não SUS: 0


## 20. Cobertura final das fontes

A cobertura temporal das bases consolidadas é comparada para identificar as competências disponíveis simultaneamente no SIH/SUS e no CNES, além dos anos que possuem população de referência validada.

In [322]:
competencias_comuns = sorted( competencias_sih & competencias_cnes)
somente_sih = sorted( competencias_sih - competencias_cnes )
somente_cnes = sorted( competencias_cnes - competencias_sih)

print( "Competências comuns SIH/CNES:", len(competencias_comuns))
print( "Primeira competência comum:", competencias_comuns[0])
print( "Última competência comum:", competencias_comuns[-1])

print( "Somente no SIH:", somente_sih )
print( "Somente no CNES:", somente_cnes)

Competências comuns SIH/CNES: 66
Primeira competência comum: (2021, '01')
Última competência comum: (2026, '06')
Somente no SIH: []
Somente no CNES: [(2026, '07')]


In [323]:
cobertura_final = pd.DataFrame(
    [
        {
            "ano": ano,
            "meses_sih": sum(
                1
                for ano_comp, _ in competencias_sih
                if ano_comp == ano
            ),
            "meses_cnes": sum(
                1
                for ano_comp, _ in competencias_cnes
                if ano_comp == ano
            ),
            "meses_comuns": sum(
                1
                for ano_comp, _ in competencias_comuns
                if ano_comp == ano
            ),
            "populacao_validada": (
                ano in anos_populacao
            ),
        }
        for ano in ANOS
    ]
)

cobertura_final

,ano,meses_sih,meses_cnes,meses_comuns,populacao_validada
0,2021,12,12,12,True
1,2022,12,12,12,True
2,2023,12,12,12,False
3,2024,12,12,12,True
4,2025,12,12,12,True
5,2026,6,7,6,False


In [324]:
print( "Total de competências comuns SIH/CNES:", len(competencias_comuns))
print( "Período comum SIH/CNES:", f"{competencias_comuns[0]} até {competencias_comuns[-1]}")
print( "Anos com população validada:", sorted(anos_populacao))
print( "Anos ainda sem população definida:", anos_sem_populacao)

Total de competências comuns SIH/CNES: 66
Período comum SIH/CNES: (2021, '01') até (2026, '06')
Anos com população validada: [2021, 2022, 2024, 2025]
Anos ainda sem população definida: [2023, 2026]


## 21. Conclusão

A consolidação multianual permitiu organizar e validar as bases do SIH/SUS, CNES e população de referência para o período analisado.

Foram identificadas 66 competências comuns entre SIH/SUS e CNES, de janeiro de 2021 a junho de 2026. As fontes populacionais estão validadas para 2021, 2022, 2024 e 2025, enquanto 2023 e 2026 permanecem pendentes de definição metodológica.

### Exportação das bases consolidadas

In [325]:
pasta_silver = Path("../data/silver")
pasta_silver.mkdir(parents=True, exist_ok=True)
df_sih_multianual.to_parquet( pasta_silver / "sih_multianual.parquet", index=False)
df_cnes_multianual.to_parquet( pasta_silver / "cnes_multianual.parquet", index=False)
df_populacao.to_parquet( pasta_silver / "populacao_multianual.parquet", index=False )
print("Bases consolidadas exportadas com sucesso.")

Bases consolidadas exportadas com sucesso.
